# 02 — Feature engineering WOE bằng SQL

Chạy `sql/features.sql` lên `data/credit.db`, rồi kiểm ba thứ: pipeline có toàn vẹn không, SQL có tính đúng không, và các feature sinh ra có chồng lấn nhau không.

Mọi phép tính nằm trong file SQL. Notebook này chỉ gọi và kiểm chứng.

Quy tắc chi phối cả bước: điểm cắt bin và giá trị WOE **chỉ được tính trên `split = 'train'`**. Chúng là tham số học được chứ không phải tiền xử lý.

In [1]:
import sys, sqlite3
from pathlib import Path
import numpy as np, pandas as pd

sys.path.insert(0, str(Path.cwd().parent / 'src'))
import config, build_features, woe_check

pd.set_option('display.width', 200); pd.set_option('display.max_columns', 50)
print('DB:', config.DB_PATH)

DB: D:\hoc-ai\learning-journey\credit_scoring\project6_credit_scoring\data\credit.db


---
## 1. Chạy SQL

`features.sql` tạo sáu bảng: `row_values` (long format) → `bin_cuts` (điểm cắt từ train) → `row_bins` (gán bin cho mọi dòng) → `woe_lookup` (bảng tra WOE/IV) → `iv_summary` → `features_woe` (bảng rộng).

SQL dùng trong đó: CTE, window function (`NTILE`, `ROW_NUMBER`, `COUNT OVER`), `GROUP BY`, `CASE`, `LEFT JOIN`, `CROSS JOIN`, và range join khi áp điểm cắt.

In [2]:
iv = build_features.run()
iv

LN(): python


,variable,n_bins,iv,muc
0,revolving_util,11,1.1354,rat manh - kiem leakage
1,late_90,6,0.8697,rat manh - kiem leakage
2,late_30_59,6,0.7687,rat manh - kiem leakage
3,late_60_89,5,0.5980,rat manh - kiem leakage
4,age,10,0.2545,trung binh
5,debt_ratio_valid,11,0.0781,yeu
6,monthly_income,12,0.0748,yeu
7,open_credit_lines,10,0.0676,yeu
8,real_estate_loans,4,0.0564,yeu
9,dependents,5,0.0364,yeu


Bốn biến vượt ngưỡng IV 0,5. Ngưỡng đó sinh ra cho application scorecard; bộ dữ liệu này là behavioral nên hành vi quá khứ dự báo hành vi tương lai rất mạnh là bình thường.

`late_90` vẫn là biến đáng soi nhất vì nó đo *đúng cùng một sự kiện* với target, chỉ khác cửa sổ thời gian. Không kiểm chứng được vì thiếu cột ngày, nên sẽ đo mức ảnh hưởng ở bước đánh giá thay vì giả định.

In [3]:
# Diem cat lay tu train, da khu trung de moi gia tri bang nhau roi cung mot bin
con = sqlite3.connect(config.DB_PATH)
cuts = pd.read_sql('SELECT * FROM bin_cuts ORDER BY variable, cut_no', con)
for v, g in cuts.groupby('variable'):
    print(f"{v:20s} {len(g)} diem cat -> {len(g)+1} bin : {[round(x,4) for x in g.upper_bound]}")

age                  9 diem cat -> 10 bin : [33.0, 39.0, 44.0, 48.0, 52.0, 56.0, 60.0, 65.0, 72.0]
debt_ratio_valid     9 diem cat -> 10 bin : [0.0211, 0.1063, 0.1741, 0.2344, 0.2923, 0.3548, 0.4285, 0.5296, 0.7244]
monthly_income       9 diem cat -> 10 bin : [2200.0, 3050.0, 3900.0, 4600.0, 5450.0, 6400.0, 7553.0, 9165.0, 11666.0]
open_credit_lines    9 diem cat -> 10 bin : [3.0, 4.0, 5.0, 6.0, 8.0, 9.0, 10.0, 12.0, 15.0]
revolving_util       9 diem cat -> 10 bin : [0.0029, 0.0191, 0.0433, 0.0828, 0.1536, 0.2699, 0.4419, 0.6927, 0.9773]


---
## 2. Kiểm tính toàn vẹn

Ba phép kiểm. Phép thứ ba quan trọng nhất: bước 6 của `features.sql` dùng `LEFT JOIN` chứ không `INNER JOIN` một cách cố ý, nên nếu có bin nào xuất hiện ở test hoặc OOT mà train chưa từng thấy thì WOE sẽ là NULL và lộ ra ngay, thay vì dòng đó biến mất khỏi kết quả trong im lặng.

In [4]:
for k, v in build_features.check().items():
    print(f'{k}: {v}')

n_features_woe: 149999
n_row_bins: 1499990
n_row_bins_ky_vong: 1499990
n_dong_woe_null: 0
ok: True


---
## 3. Đối chiếu SQL với pandas

`src/woe_check.py` cài đặt lại WOE/IV bằng pandas, không đọc lại bảng nào do SQL tạo ra ngoài `applications`. Nếu cả hai cùng sai thì phải sai theo hai đường khác nhau. Đối chiếu với chính mình thì không chứng minh được gì.

In [5]:
for k, v in woe_check.compare().items():
    print(f'{k}: {v}')

so_bin_sql: 80
so_bin_pandas: 80
khop_het_bin: True
lech_so_dong: 0
lech_so_bad: 0
lech_woe_lon_nhat: 4.912280613944553e-07
lech_iv_lon_nhat: 0.0
so_bien_lech_iv: 0


Lần chạy đầu tiên phép kiểm này bắt được một lỗi trong chính nó, và lỗi đó đáng ghi lại.

Bản pandas ban đầu tính tổng good/bad từ bảng **long** thay vì bảng gốc. Bảng long có 10 dòng cho mỗi dòng gốc (một dòng mỗi biến) nên tổng bị gấp 10 lần, kéo theo `pct_good` và `pct_bad` đều bị chia 10.

Vì `WOE = ln(pct_good / pct_bad)` là một **tỉ số**, sai số ở mẫu số chung triệt tiêu hoàn toàn và WOE vẫn khớp tới 4,9e-07. Nhưng `IV = Σ(pct_good − pct_bad)·WOE` dùng **hiệu**, nên IV sai đúng 10 lần.

Bài học: WOE khớp không chứng minh IV đúng. Một phép kiểm không thể fail ở thứ mình cần kiểm thì không phải phép kiểm.

---
## 4. Bốn biến không đơn điệu

Đây là chỗ quyết định cách chia bin. Ràng buộc đơn điệu là chuẩn mực của scorecard, nhưng ép nó lên một quan hệ thật sự không đơn điệu là xoá tín hiệu.

In [6]:
woe = pd.read_sql('SELECT * FROM woe_lookup ORDER BY variable, bin', con)
for v in ['open_credit_lines', 'debt_ratio_valid', 'revolving_util', 'real_estate_loans']:
    t = woe[woe.variable == v][['bin','n','n_bad','bad_rate_pct','woe','iv_part']]
    print(f"\n=== {v}  (IV = {iv.loc[iv.variable==v,'iv'].iloc[0]}) ===")
    print(t.to_string(index=False))


=== open_credit_lines  (IV = 0.0676) ===
bin     n  n_bad  bad_rate_pct       woe  iv_part
 01 15417   1661        10.774 -0.522240 0.050283
 02  8131    528         6.494  0.030907 0.000073
 03  9042    566         6.260  0.070105 0.000411
 04  9490    527         5.553  0.197364 0.003234
 05 18041    950         5.266  0.253550 0.009905
 06  7976    469         5.880  0.136693 0.001338
 07  6734    412         6.118  0.094472 0.000550
 08 10744    640         5.957  0.122923 0.001466
 09  9754    617         6.326  0.058923 0.000314
 10  9670    648         6.701 -0.002765 0.000001

=== debt_ratio_valid  (IV = 0.0781) ===
      bin     n  n_bad  bad_rate_pct       woe  iv_part
       01  8303    408         4.914  0.326422 0.007325
       02  8303    577         6.949 -0.041791 0.000141
       03  8302    539         6.492  0.031113 0.000076
       04  8303    500         6.022  0.111360 0.000935
       05  8301    416         5.011  0.305737 0.006481
       06  8303    451         

Bốn biến này không đơn điệu, nhưng **không cùng một hình dạng**. Chỉ `open_credit_lines` và `real_estate_loans` là chữ U thật. `revolving_util` tăng đơn điệu từ bin 2 trở đi và chỉ có một cái móc ở bin đầu (2,55% so với 1,27%). `debt_ratio_valid` thì bin thấp nhất lại là bin **an toàn nhất** (4,91%, thấp nhất bảng), nên đầu thấp của biến không phải một đầu rủi ro; cái không đơn điệu ở đây là một dao động trong nửa dưới.

Cơ chế đằng sau chúng cũng khác nhau, chi tiết trong `results/iv_report.md` §2. Tóm tắt: nhánh trái của `open_credit_lines` đúng là nhóm bị hạn chế tín dụng (2 hạn mức, thu nhập trung vị 3.333, utilization 0,439 so với 0,137 ở đáy), nhánh phải thì thu nhập cao nhất bảng và không hề bị hạn chế. `real_estate_loans` hai đầu là hai nhóm thu nhập trái ngược, 3.967 và 9.032. Còn cái móc ở `revolving_util` **không phải hồ sơ mỏng**: nhóm đó có trung vị 6 hạn mức đang mở và không ai có 0 hạn mức; khác biệt thật là tiền sử trễ hạn, 12,4% so với 8,6%.

Quyết định giữ nguyên hình dạng dựa trên 5-fold CV **bên trong train**, so sánh theo cặp trên cùng fold. Chênh lệch Gini `giữ nguyên hình − ép đơn điệu`, ép đúng chiều nghiệp vụ: `revolving_util` +0,0063, `debt_ratio_valid` +0,0146, `open_credit_lines` +0,0176, `real_estate_loans` +0,0205, cả bốn cùng dấu ở cả 5 fold. `monthly_income` hoà (+0,0003), `age` vốn đã đơn điệu. Với hai biến chữ U, ép SAI chiều tốn gấp 6 đến 7 lần: `open_credit_lines` +0,1257 và `real_estate_loans` +0,1005. Bảng đầy đủ ở `results/iv_report.md` §2, kèm ghi chú vì sao bảng đã phải chạy lại.

Lý do giữ là kết quả CV chứ không phải vì kể được câu chuyện nghiệp vụ, mà với `revolving_util` thì tôi chưa kể được. Cái giá là mã lý do khi từ chối hồ sơ phải gắn vào **bin** thay vì vào **biến**, vì một biến chữ U cần hai chiều lý do khác nhau.

---
## 5. Các feature có chồng lấn nhau không

Ba biến `late_*` dùng chung bin sentinel, và `debt_ratio_valid` có bin `X_INVALID` đúng bằng hợp của `X_MISSING` và `X_ZERO` bên `monthly_income`. Cần đo xem chuyện đó có thành đa cộng tuyến không.

In [7]:
f = pd.read_sql("SELECT * FROM features_woe WHERE split='train'", con)
cols = [c for c in f.columns if c.startswith('woe_')]
corr = f[cols].corr()

X = (f[cols].values - f[cols].values.mean(0)) / f[cols].values.std(0)
vif = np.diag(np.linalg.inv(np.corrcoef(X, rowvar=False)))
print('VIF:')
for name, v in sorted(zip(cols, vif), key=lambda x: -x[1]):
    print(f'  {name:24s} {v:6.2f}')
print(f'\nTuong quan cap lon nhat: {corr.where(~np.eye(len(cols),dtype=bool)).abs().max().max():.3f}')
corr.round(2)

VIF:
  woe_revolving_util         1.30
  woe_late_90                1.25
  woe_late_60_89             1.23
  woe_late_30_59             1.21
  woe_open_credit_lines      1.19
  woe_debt_ratio             1.19
  woe_age                    1.17
  woe_monthly_income         1.14
  woe_real_estate_loans      1.14
  woe_dependents             1.13

Tuong quan cap lon nhat: 0.353


,woe_revolving_util,woe_age,woe_debt_ratio,woe_monthly_income,woe_open_credit_lines,woe_late_30_59,woe_late_60_89,woe_late_90,woe_real_estate_loans,woe_dependents
woe_revolving_util,1.00,0.29,0.15,0.12,0.21,0.26,0.21,0.28,0.09,0.11
woe_age,0.29,1.00,0.10,0.07,0.06,0.10,0.09,0.10,0.01,0.27
woe_debt_ratio,0.15,0.10,1.00,0.26,-0.15,0.09,0.03,0.00,-0.15,0.13
woe_monthly_income,0.12,0.07,0.26,1.00,0.11,0.03,0.05,0.08,0.14,-0.07
woe_open_credit_lines,0.21,0.06,-0.15,0.11,1.00,-0.01,0.05,0.15,0.29,-0.12
woe_late_30_59,0.26,0.10,0.09,0.03,-0.01,1.00,0.33,0.29,0.00,0.07
woe_late_60_89,0.21,0.09,0.03,0.05,0.05,0.33,1.00,0.35,0.04,0.04
woe_late_90,0.28,0.10,0.00,0.08,0.15,0.29,0.35,1.00,0.09,0.03
woe_real_estate_loans,0.09,0.01,-0.15,0.14,0.29,0.00,0.04,0.09,1.00,-0.13
woe_dependents,0.11,0.27,0.13,-0.07,-0.12,0.07,0.04,0.03,-0.13,1.00


VIF cao nhất 1,30, tương quan cặp lớn nhất 0,35. Không có vấn đề.

---
## Xong bước này

| | |
|---|---|
| `bin_cuts` | điểm cắt từ `NTILE(10)` trên train, khử trùng để không cắt ngang giá trị bằng nhau |
| `row_bins` | 1.499.990 dòng = 149.999 × 10 biến |
| `woe_lookup` | 80 bin, WOE/IV tính trên train |
| `features_woe` | 149.999 dòng × 10 cột WOE, không dòng nào NULL |
| Đối chiếu pandas | lệch WOE 4,9e-07, lệch IV 0,0 |
| VIF | cao nhất 1,30 |

Bước tiếp theo: hồi quy logistic trên các cột WOE này, kiểm hệ số cùng dấu và độ lớn quanh 1, rồi quy đổi ra thang điểm theo PDO và base-odds.